In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, lit, floor, rand, concat, explode
from pyspark.sql import functions as F
import time

In [2]:
spark = SparkSession.builder.appName("AQE_vs_Salting").getOrCreate()

In [3]:
# Benchmark helper
def benchmark(name, func):
    start = time.time()
    result = func()
    end = time.time()
    print(f"{name} → count: {result}, duration: {end - start:.2f} s")
    return result

In [4]:
# Dataset 1: Moderate skew (AQE-friendly)
N = 5_000_000
moderate = spark.range(0, N).withColumn(
    "user_id",
    when(col("id") < N * 0.05, lit(1))  # 5% hot key
    .otherwise((col("id") % 100))
)

dim1 = spark.range(0, 100).withColumnRenamed("id", "user_id")

In [5]:
# AQE join
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionFactor", 5)
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", 64*1024*1024)

benchmark(
    "Moderate skew join with AQE",
    lambda: moderate.join(dim1, "user_id").count()
)

Moderate skew join with AQE → count: 5000000, duration: 1.88 s


5000000

In [6]:
# Dataset 2: Extreme skew (manual deterministic salting)
extreme = spark.range(0, N).withColumn(
    "user_id",
    when(col("id") < N * 0.4, lit(1))  # 40% hot key
    .otherwise((col("id") % 100))
)

dim2 = spark.range(0, 100).withColumnRenamed("id", "user_id")

In [7]:
# Manual deterministic salting
SALT_BUCKETS = 16

# Large table: add random salt
salted_extreme = extreme.withColumn(
    "salt",
    floor(rand() * SALT_BUCKETS)
).withColumn(
    "salted_key",
    col("user_id") * SALT_BUCKETS + col("salt")
)

In [8]:
# Small table exploded on all salts
salt_values = spark.range(SALT_BUCKETS).withColumnRenamed("id", "salt")
dim_salted = dim2.crossJoin(salt_values).withColumn(
    "salted_key",
    col("user_id") * SALT_BUCKETS + col("salt")
)

benchmark(
    "Extreme skew join with manual deterministic salting",
    lambda: salted_extreme.join(dim_salted, "salted_key").count()
)

Extreme skew join with manual deterministic salting → count: 5000000, duration: 0.60 s


5000000

In [10]:
spark = SparkSession.builder.appName("Skew_Join_Comparison").getOrCreate()

# Dataset 1: Moderate skew (AQE-friendly)
N = 500_000_000
moderate = spark.range(0, N).withColumn(
    "user_id",
    when(col("id") < N * 0.05, lit(1))  # 5% hot key
    .otherwise((col("id") % 100))
)
dim1 = spark.range(0, 100).withColumnRenamed("id", "user_id")

# Dataset 2: Extreme skew (Manual salting required)
extreme = spark.range(0, N).withColumn(
    "user_id",
    when(col("id") < N * 0.4, lit(1))  # 40% hot key
    .otherwise((col("id") % 100))
)
dim2 = spark.range(0, 100).withColumnRenamed("id", "user_id")

# No mitigation joins
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "false")
print("\n--- NO mitigation AQE off---")
benchmark("Moderate skew join (no mitigation)", lambda: moderate.join(dim1, "user_id").count())
benchmark("Extreme skew join (no mitigation)", lambda: extreme.join(dim2, "user_id").count())

SALT_BUCKETS = 16

def salted_join(fact_df, dim_df, salt_buckets=SALT_BUCKETS):
    # Large table: add random salt
    salted_fact = fact_df.withColumn(
        "salt",
        floor(rand() * salt_buckets)
    ).withColumn(
        "salted_key",
        col("user_id") * salt_buckets + col("salt")
    )
    # Small table: explode all salt values
    salt_values = spark.range(salt_buckets).withColumnRenamed("id", "salt")
    salted_dim = dim_df.crossJoin(salt_values).withColumn(
        "salted_key",
        col("user_id") * salt_buckets + col("salt")
    )
    return salted_fact.join(salted_dim, "salted_key").count()

print("\n--- Manual deterministic salting AQE OFF ---")
benchmark("Moderate skew join (manual salting)", lambda: salted_join(moderate, dim1))
benchmark("Extreme skew join (manual salting)", lambda: salted_join(extreme, dim2))

# AQE skew joins
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionFactor", 5)
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", 64*1024*1024)

print("\n--- NO mitigation AQE on---")
benchmark("Moderate skew join (no mitigation)", lambda: moderate.join(dim1, "user_id").count())
benchmark("Extreme skew join (no mitigation)", lambda: extreme.join(dim2, "user_id").count())

print("\n--- AQE skew join ---")
benchmark("Moderate skew join (AQE)", lambda: moderate.join(dim1, "user_id").count())
benchmark("Extreme skew join (AQE)", lambda: extreme.join(dim2, "user_id").count())

# Manual deterministic salting
SALT_BUCKETS = 16

def salted_join(fact_df, dim_df, salt_buckets=SALT_BUCKETS):
    # Large table: add random salt
    salted_fact = fact_df.withColumn(
        "salt",
        floor(rand() * salt_buckets)
    ).withColumn(
        "salted_key",
        col("user_id") * salt_buckets + col("salt")
    )
    # Small table: explode all salt values
    salt_values = spark.range(salt_buckets).withColumnRenamed("id", "salt")
    salted_dim = dim_df.crossJoin(salt_values).withColumn(
        "salted_key",
        col("user_id") * salt_buckets + col("salt")
    )
    return salted_fact.join(salted_dim, "salted_key").count()

print("\n--- Deterministic salting AQE on ---")
benchmark("Moderate skew join (deterministic salting)", lambda: salted_join(moderate, dim1))
benchmark("Extreme skew join (deterministic salting)", lambda: salted_join(extreme, dim2))



--- NO mitigation AQE off---
Moderate skew join (no mitigation) → count: 500000000, duration: 1.25 s
Extreme skew join (no mitigation) → count: 500000000, duration: 1.10 s

--- Manual deterministic salting AQE OFF ---
Moderate skew join (manual salting) → count: 500000000, duration: 1.76 s
Extreme skew join (manual salting) → count: 500000000, duration: 1.77 s

--- NO mitigation AQE on---
Moderate skew join (no mitigation) → count: 500000000, duration: 1.15 s
Extreme skew join (no mitigation) → count: 500000000, duration: 1.12 s

--- AQE skew join ---
Moderate skew join (AQE) → count: 500000000, duration: 1.19 s
Extreme skew join (AQE) → count: 500000000, duration: 1.16 s

--- Deterministic salting AQE on ---
Moderate skew join (deterministic salting) → count: 500000000, duration: 1.76 s
Extreme skew join (deterministic salting) → count: 500000000, duration: 1.84 s


500000000